In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:

import tensorflow as tf
from src.utils.dataframe_utils import create_dataframe


df = create_dataframe("../data/train.csv", "../data/boneage-training-dataset", segmented=False)
df_val = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset", segmented=False)

In [3]:
from src.preprocessing.scaling import scaling_data
dataset, dataset_val, scaler = scaling_data(df, df_val)

In [4]:
from src.preprocessing.data_loader import load_image, create_dataset_tf

dataset = create_dataset_tf(dataset, load_image, num_samples=2000)
dataset_val = create_dataset_tf(dataset_val, load_image)


In [5]:
# define a simple CNN for regression (e.g. bone age)
from src.model.cnn import CNN
model = CNN()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,993 (394.50 KB)

 Trainable params: 100,993 (394.50 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
#to tune 
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [7]:
history = model.fit(
    dataset,
    validation_data=dataset_val,
    epochs=50,
    callbacks=[early_stop]
)


Epoch 1/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 25s 349ms/step - loss: 1.0087 - mae: 0.8202 - val_loss: 1.1105 - val_mae: 0.8528
Epoch 2/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 362ms/step - loss: 1.0141 - mae: 0.8293 - val_loss: 1.1053 - val_mae: 0.8543
Epoch 3/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 28s 444ms/step - loss: 1.0348 - mae: 0.8269 - val_loss: 1.1084 - val_mae: 0.8567
Epoch 4/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 28s 430ms/step - loss: 0.9665 - mae: 0.8072 - val_loss: 1.1263 - val_mae: 0.8408
Epoch 5/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 26s 408ms/step - loss: 0.9766 - mae: 0.8090 - val_loss: 1.1044 - val_mae: 0.8443
Epoch 6/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 27s 420ms/step - loss: 0.9755 - mae: 0.8014 - val_loss: 1.0709 - val_mae: 0.8367
Epoch 7/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 28s 433ms/step - loss: 0.9595 - mae: 0.8010 - val_loss: 1.0412 - val_mae: 0.8076
Epoch 8/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 26s 399ms/step - loss: 0.9130 - mae: 0.7732 - val_loss: 1.0208 - val_mae: 0.7921
Epoch 9/50
63/63 ━━━━━━━━━━━━━━━━━━━━ 26s 414ms/

In [ ]:
import pickle

with open("training_history.pkl", "wb") as f:
    pickle.dump(history.history, f)

model.save("modello.keras")


In [9]:
model = tf.keras.models.load_model("modello.keras")


In [ ]:
import pickle

with open("history.pkl", "rb") as f:
    history_dict = pickle.load(f)

In [ ]:
from src.visualization.plots import plot_loss
plot_loss(history)